# BEHAVIOR-1K: A Human-Centered, Embodied AI Benchmark with 1,000 Everyday Activities and Realistic Simulation

**Paper:** Li, Zhang, Wong, Gokmen, Srivastava, Martín-Martín, Wang, Levine, Ai, et al. (Stanford, UT Austin, UIUC, USC, Salesforce) — *BEHAVIOR-1K: A Human-Centered, Embodied AI Benchmark with 1,000 Everyday Activities and Realistic Simulation*, arXiv:2403.09227 (a preliminary version appeared as BEHAVIOR-100 at CoRL 2022)

**Links:** [Paper](https://arxiv.org/abs/2403.09227) | [Project website](https://behavior.stanford.edu) | [Code (BEHAVIOR-1K / OmniGibson)](https://github.com/StanfordVL/BEHAVIOR-1K) | [Docs](https://behavior.stanford.edu/omnigibson/getting_started/installation.html)

---

### Table of contents

1. What problem is this solving, and why a survey?
2. Theory — the human-preference survey methodology
3. Theory — the BEHAVIOR-1K Dataset: knowledge base annotation pipeline
4. Theory — BDDL: predicate-logic activity definitions, and what's new here vs. LIBERO's BDDL
5. Theory — OmniGibson: the simulator, and why a new one was needed
6. Theory — how BEHAVIOR-1K compares to other embodied AI benchmarks
7. Theory — the three evaluated baselines: RL-VMC, RL-Prim., RL-Prim.Hist.
8. Headline experimental results, with caveats
9. Theory — the sim-to-real study
10. Installation — and an important scope-setting note before you try
11. Code: exploring the WordNet-derived synset/property hierarchy
12. Code: writing and validating a toy BDDL-style activity definition
13. Code: a minimal initial-condition sampler and goal checker
14. Code: action-primitive interface sketch (pick / place / navigate / wipe)
15. Code: reproducing the survey's diversity statistic (Gini index) on toy data
16. BEHAVIOR-1K vs. LIBERO vs. CALVIN vs. RLBench
17. Discussion questions
18. References

### Why this paper, in one paragraph

Most embodied-AI benchmarks are designed by researchers picking tasks they find interesting or tractable. BEHAVIOR-1K inverts that: it starts from a **1,461-person survey** asking ordinary people what they actually want a robot to do for them, and only *then* builds simulation infrastructure realistic enough to support the resulting activity list. That list turns out to be dominated by unglamorous chores — scrubbing floors, cleaning bathtubs — not the pick-and-place tasks most manipulation research targets, and supporting it required building a new simulator (**OmniGibson**, on top of NVIDIA Omniverse/PhysX 5) because no existing one could simulate fluids, deformables, and thermal effects at the needed scale. If you take one structural idea from this paper into your own project-scoping, it should be this: **the benchmark's task distribution is an empirical finding, not a design choice** — worth remembering the next time you or a paper you're reading picks "interesting" tasks by hand.

> **This is the technical revision** of this tutorial. It replaces several illustrative/toy explanations with the paper's actual formalism, pulled from the appendix: the exact PPO/SAC objectives and hyperparameters (Sec. 7.1), the real network architecture (Sec. 7.2), the exact logical-predicate checking and sampling functions with their numeric thresholds (Sec. 5.1, 13), the real BDDL substance/three-valued-predicate/composition mechanisms (Sec. 4.1), and the paper's actual annotation-quality and simulator-performance measurements (Sec. 3.1, 10.1). Toy/illustrative code is kept only where the real mechanism genuinely requires a live OmniGibson install to demonstrate, and is labeled as such.

## 0. Notebook setup

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

plt.style.use("dark_background")
mpl.rcParams.update({
    "figure.facecolor": "#1e1e1e",
    "axes.facecolor": "#1e1e1e",
    "savefig.facecolor": "#1e1e1e",
    "axes.edgecolor": "#888888",
    "axes.labelcolor": "#eeeeee",
    "xtick.color": "#cccccc",
    "ytick.color": "#cccccc",
    "text.color": "#eeeeee",
    "grid.color": "#3a3a3a",
    "font.size": 11,
    "figure.figsize": (7, 4),
})

SEED = 0
np.random.seed(SEED)
print("Setup complete.")

## 1. What problem is this solving, and why a survey?

Benchmarks like RLBench, Meta-World, and CALVIN are diverse in *skills* (grasping, insertion, chaining instructions) but their task lists come from researcher intuition about what's interesting or achievable. BEHAVIOR-1K's authors argue this leaves an open question: are these the tasks people actually want automated? Their answer is to source activities from **time-use surveys** (which record how people actually spend their day — American, European, and Multinational Time Use Surveys) and **WikiHow** (180,000+ how-to articles, capturing important-but-infrequent activities time-use surveys miss), then ask a representative sample how much they'd want each one delegated to a robot. The resulting ranked list is what defines "BEHAVIOR-1K" — the benchmark's *name* literally refers to the top 1,000 activities by this human-preference score (909 newly ranked plus 91 carried over from the predecessor BEHAVIOR-100 benchmark).

## 2. Theory: the human-preference survey methodology

### 2.1 Sourcing and filtering

- Combine three time-use surveys $\to$ ~540 candidate activities.
- Add WikiHow article titles (180,000+ articles) for activities that matter but aren't frequent enough to show up in time-use data.
- **Filter for simulation feasibility** *before* surveying, using explicit exclusion criteria — this is a design choice worth noticing: the benchmark doesn't survey people about tasks it could never simulate, which avoids wasting survey budget but also means the "human need" signal is already conditioned on "things a simulator plausibly could support."

| Filtering principle | Example excluded activity |
|---|---|
| Requires unsimulated physics/chemistry | steaming clothes, making soap |
| Involves creating/consuming media | reading a book |
| Requires more than a day of real time | drying seeds overnight |
| Requires non-visual perception | sweetening food (taste) |
| Needs geometric precision beyond BDDL | setting up a nativity scene |
| Predicated on specific branded items | using a specific spray-cleaner brand |
| Involves other people or live animals | asking for a raise |

This leaves 2,090 candidate activities that actually get surveyed.

### 2.2 Survey instrument and design choices

- **1,461 respondents** on Amazon Mechanical Turk, 50 independent Likert-scale (1–10) responses per activity ("rate how much you want a robot to do this activity for you").
- The authors piloted both the **wording** ("robot" vs. "assistant" vs. "automation" — no significant difference found across 30 pairwise t-tests) and the **response format** (10-point Likert vs. three-element best-worst scaling — strongly correlated via Kendall's tau, so they kept the cheaper Likert format). This is a small but instructive piece of survey-methodology hygiene: they tested whether their measurement instrument's arbitrary choices actually mattered before committing to one, rather than assuming a scale choice is inconsequential.
- Quality control: repeated attention-check questions (reject if paired responses differ by >2 points on more than one pair) and rejection of respondents with no variance across their 50 ratings.

### 2.3 What the survey found

- Scores ranged 1.9–9.3 across activities (mean 5.16) with a **Gini index of 0.158** — moderate but real dispersion, meaning people's desire to automate different chores is genuinely uneven, not flat.
- **Tedious, physically unpleasant chores score highest** (e.g. scrubbing floors, cleaning a bathtub); **recreational activities score lowest** (e.g. game-play). Roughly 200 cleaning activities and 200+ cooking activities appear among the top-ranked set.
- BEHAVIOR-1K = the **top 1,000** activities by this score (909 new + 91 carried from BEHAVIOR-100).

We'll reproduce the Gini-index diversity calculation on toy data in Section 15 — it's a genuinely useful summary statistic to have in your toolkit any time you want to characterize how "spread out" a ranked preference distribution is, well beyond this one paper.

In [ ]:
# Illustrative reconstruction of the survey's category breakdown (Section 2.3):
# the paper states there are ~200 cleaning activities and 200+ cooking activities
# among the top-ranked set, with many other categories represented. Exact per-category
# counts aren't published, so treat the chart below as illustrative proportions, not
# extracted figures -- useful for building intuition about the shape of the result.
categories = ["Cleaning", "Cooking", "Organizing", "Yard/outdoor", "Laundry", "Shopping/errands", "Other"]
illustrative_counts = [200, 210, 140, 110, 90, 80, 170]  # illustrative, sums to ~1000

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(categories, illustrative_counts, color="#66bb6a")
ax.set_xlabel("# activities in BEHAVIOR-1K (illustrative)")
ax.set_title("Illustrative activity-category breakdown\n(cleaning & cooking dominate, as reported)")
ax.invert_yaxis()
for bar, count in zip(bars, illustrative_counts):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2, str(count),
            va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Theory: the BEHAVIOR-1K Dataset — knowledge base annotation pipeline

Having a ranked activity *list* is only step one. Each activity needs a machine-checkable definition (objects involved, their properties, initial/goal conditions) before a simulator can instantiate or evaluate it. The annotation pipeline that produces this is worth understanding in detail because it's a reusable pattern for "how do you turn 1,000 fuzzy natural-language activity names into 1,000 machine-checkable task specifications without 1,000 person-hours of expert annotation":

**Step 1 — object discovery via WikiHow.** For each activity, crowdworkers collect 5 WikiHow articles describing it. A noun-phrase chunking model extracts candidate objects from the article text, which are then manually filtered down to tangible physical objects. This grounds the object vocabulary in how people actually describe doing the activity, rather than an annotator guessing what objects "should" be involved.

**Step 2 — synset mapping.** Every extracted object noun phrase is mapped by crowdworkers to a **WordNet synset** (a disambiguated word sense, e.g. `apple.n.01`), eliminating ambiguity ("bat" the animal vs. "bat" the sports equipment) and — crucially — placing every object into WordNet's **hierarchical** structure. Where no WordNet synset fits, a custom synset is created. This yields 1,538 WordNet leaf synsets + 1,426 custom synsets = **2,964 leaf-level synsets** total.

**Step 3 — property annotation.** Each leaf synset gets annotated with which of ~35 simulatable object properties apply to it (`cookable`, `sliceable`, `heatSource`, `fillable`, `cloth`, `liquid`, ...). Annotation method varies by property: some properties are inferred *programmatically* from other properties (e.g. `deformable` = union of `softBody`/`cloth`/`rope`), some are assigned by **GPT-3** (validated against human-annotated ground truth, used only for properties where GPT-3's Hamming distance and false-positive rate were both under 10%), and the rest by human crowdworkers or the researchers directly (for properties too simulator-specific to delegate).

**Step 4 — hierarchical propagation.** Properties are annotated at the **leaf** level only, then propagated *up* the WordNet hierarchy so higher-level synsets (e.g. `container.n.01`) can be referenced directly in an activity definition (e.g. "5 `edible_fruit.n.01`s" instead of enumerating every fruit species) — while avoiding unsolvable definitions, a non-leaf synset's properties are defined as the **intersection** of all its descendants' properties, so referencing a general category never silently promises a property that some concrete instance of that category can't actually provide.

**Step 5 — property *parameters*.** Some properties need object-specific numeric parameters, not just a boolean flag — a `cookable` object needs a **cook temperature**, a `heatSource` needs a **temperature it generates**. These are annotated per object, e.g.:

| Object | cook temperature (°C) |
|---|---|
| crab | 63 |
| squash | 58 |
| chicken leg | 74 |

| Heat source | temperature generated (°C) |
|---|---|
| toaster oven | 204 |
| coffee maker | 93 |
| ember | 1093 |

**Step 6 — transition rules.** Some processes are too complex to fully physically simulate (blending fruit into a smoothie, sanding a rusted surface) but the *actions* leading to them are simulatable (placing fruit in a blender). A **Transition Machine** specifies these as symbolic rules: when a set of conditions holds among a group of objects, swap them for the "after" state. This lets the benchmark support activities whose *physical process* is out of reach while still requiring the robot to perform the actual manipulation that would trigger it.

The result: a knowledge base of tens of thousands of annotated elements underlying 1,000 activity definitions, quality-checked by five experienced ML annotators against a subset of every annotation type, achieving >96.8% approval.

**Visualizing the pipeline** — six stages, each feeding the next, turning a ranked activity name into a machine-checkable BDDL file:

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

stages = [
    "1. WikiHow article\ncollection + noun-\nphrase extraction",
    "2. WordNet synset\nmapping\n(2,964 leaf synsets)",
    "3. Property\nannotation\n(GPT-3 / crowdworkers)",
    "4. Hierarchical\npropagation\n(intersection rule)",
    "5. Property\nparameter annotation\n(temperatures, etc.)",
    "6. Transition rules\n+ BDDL activity\ndefinition",
]
colors = ["#4fc3f7", "#4fc3f7", "#ffca28", "#ffca28", "#ef5350", "#ab47bc"]

fig, ax = plt.subplots(figsize=(13, 2.6))
box_w, box_h, gap = 1.9, 1.4, 0.35
x = 0.2
for i, (stage, color) in enumerate(zip(stages, colors)):
    box = mpatches.FancyBboxPatch((x, 0.3), box_w, box_h, boxstyle="round,pad=0.05",
                                    linewidth=1.2, edgecolor="#cccccc", facecolor=color, alpha=0.85)
    ax.add_patch(box)
    ax.text(x + box_w / 2, 0.3 + box_h / 2, stage, ha="center", va="center",
            fontsize=8.3, color="#111111", wrap=True)
    if i < len(stages) - 1:
        arrow = FancyArrowPatch((x + box_w, 0.3 + box_h / 2), (x + box_w + gap, 0.3 + box_h / 2),
                                 arrowstyle="-|>", mutation_scale=14, color="#eeeeee")
        ax.add_patch(arrow)
    x += box_w + gap

ax.set_xlim(0, x)
ax.set_ylim(0, 2.1)
ax.axis("off")
ax.set_title("BEHAVIOR-1K knowledge-base annotation pipeline", fontsize=12, pad=10)
plt.tight_layout()
plt.show()

### 3.1 Annotation pipeline quality metrics (Table A.2 / A.3, exact)

The "quality-checked by five experienced annotators" claim from Section 3 has actual numbers behind it. Five annotators with data-labeling/coding backgrounds re-verified a sample of every annotation stage:

| Stage | Accuracy (approval rate) | F1-score | False Discovery Rate | False Positive Rate |
|---|---|---|---|---|
| Article collection | 0.974 | 0.984 | 0.026 | — |
| Object extraction | 0.968 | 0.990 | 0.032 | — |
| Human-annotated properties | 0.990 | 0.912 | 0.031 | 0.002 |
| GPT-3 / machine-annotated properties | 0.988 | 0.930 | 0.029 | 0.003 |

$F_1 = \dfrac{2}{\text{precision}^{-1} + \text{recall}^{-1}}$, $\text{FDR} = \dfrac{FP}{TP+FP}$, $\text{FPR} = \dfrac{FP}{TN+FP}$.

The activity-definition stage itself (the final BDDL files) was rated on a 1–5 Likert scale across four questions, by independent reviewers who did not write the definitions:

| Question | Mean rating | Std. dev. |
|---|---|---|
| Q1: Are the listed objects relevant to the activity? | 4.875 | 0.331 |
| Q2: Do the initial object placements make sense? | 4.942 | 0.234 |
| Q3: Are the goal actions relevant to the activity? | 4.967 | 0.364 |
| Q4: Is this definition reasonable overall? | 4.975 | 0.156 |

**The detail worth noticing:** GPT-3-annotated properties (0.988 accuracy, 0.930 F1) are *not* meaningfully worse than human-annotated ones (0.990 accuracy, 0.912 F1) on this sample — the paper's stated criterion for using GPT-3 at all was Hamming distance and false-positive rate both under 10% against human ground truth (Section 3), and these numbers confirm that bar was actually met, not just asserted. If you're deciding whether to trust an LLM-annotated field anywhere in your own pipeline, this two-number gate (an error-rate ceiling checked *before* deployment, then re-verified *after* on a held-out sample) is a reusable pattern, not specific to BEHAVIOR-1K.

In [ ]:
stages = ["Article\ncollection", "Object\nextraction", "Human-annotated\nproperties", "GPT-3-annotated\nproperties"]
accuracy = [0.974, 0.968, 0.990, 0.988]
f1 = [0.984, 0.990, 0.912, 0.930]

x = np.arange(len(stages))
width = 0.35
fig, ax = plt.subplots(figsize=(8.5, 4.3))
ax.bar(x - width/2, accuracy, width, label="Accuracy (approval rate)", color="#4fc3f7")
ax.bar(x + width/2, f1, width, label="F1-score", color="#ffca28")
ax.set_xticks(x)
ax.set_xticklabels(stages, fontsize=9)
ax.set_ylim(0.85, 1.0)
ax.set_title("Table A.2 reproduced: annotation-pipeline quality by stage")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Theory: BDDL, and what's new here versus LIBERO's BDDL

If you built the LIBERO tutorial notebook, you already know BDDL as a predicate-logic language for declaring objects, regions, initial-state predicates, and goal predicates. BEHAVIOR-1K **inherits** BDDL from its own predecessor (BEHAVIOR-100) — LIBERO in turn *borrowed* BDDL from this lineage — but BEHAVIOR-1K's activities are dramatically more complex (long-horizon household chores vs. LIBERO's single tabletop pick-and-place goals), so the paper introduces genuinely new BDDL features:

**Representation of substances.** Household activities routinely involve things that aren't discrete rigid objects at all — water, flour, ketchup. BDDL now represents these as **substances**, split into `visualSubstance` (particles rendered but not physically simulated individually, e.g. spice dust) and `physicalSubstance`/fluids (individually or in aggregate physically simulated, e.g. pourable liquid). This is what makes activities like "clean the table with a soaked cloth" or "pour water into a cup" expressible at all.

**Three-valued predicates.** Standard predicate logic is two-valued (true/false). BEHAVIOR-1K's BDDL adds a **third value — "unknown"/don't-care** — for predicates where an activity's definition genuinely doesn't constrain a state. This matters because it lets an activity definition stay accurate without forcing annotators to arbitrarily commit to true or false for every predicate on every object; a leftover, unconstrained predicate can honestly be marked irrelevant rather than incorrectly pinned down.

**Composition and decomposition of objects.** Real cooking and cleaning activities involve objects that *become* other objects — a whole onion is diced into pieces; several ingredients combine into a dish. BDDL's composition/decomposition support lets an activity's goal condition reference the *result* of combining or slicing objects rather than requiring every possible intermediate physical configuration to be separately enumerated.

**Why this matters pedagogically:** LIBERO's BDDL tasks are all "is object X at goal region Y" — a single, static, checkable geometric condition. BEHAVIOR-1K's activities require the *symbolic language itself* to grow new primitives (substances, three-valued logic, composition) to remain expressive at household-chore complexity. If you're ever designing your own task-definition language for a new benchmark, this progression — start minimal, extend only when a concrete unrepresentable activity forces your hand — is the right order of operations, and it's visible directly in how these two benchmarks' BDDL dialects differ.

### 4.1 The exact mechanisms behind BDDL's three new features

Section 4 named the three new BDDL features at a high level. Here's how each is actually implemented — this is the level of detail you'd need to reproduce or extend BDDL yourself, not just describe it.

**Substances — the "at most one instance" rule.** Standard PDDL/BDDL objects have clean instance boundaries: `apple.n.01_1` and `apple.n.01_2` are unambiguously different apples. Substances like `orange_juice.n.01` have no such boundary — if you pour two "instances" of orange juice together, there is no longer a fact of the matter about which particle belonged to which instance, so a goal like `exists(orange_juice.n.01)(filled(orange_juice.n.01, glass_1))` becomes genuinely unsatisfiable in a well-defined sense (the juice that fills the glass might be a mix of both instances, satisfying neither alone). BEHAVIOR-1K's fix: **enforce at most one instance of any given substance per activity definition.** Quantity is still controllable — the annotator can spawn that one substance instance into as many containers as they like — but a single substance is a single simulated particle population, sidestepping the instance-mixing problem entirely. Where an annotator only needs a *container* (e.g. an empty orange juice bottle in a `PuttingAwayGroceries` activity, never referencing the liquid itself), a parallel container-only synset is provided so the (computationally expensive) particle simulation isn't spawned unnecessarily.

**Three-valued predicates — via De Morgan's law, not three-valued logic.** The underlying BDDL solver stays strictly two-valued (Boolean). What changes is which predicate a negation gets rewritten to. For predicate pairs whose negations coincide in natural language but not in the physics — `filled`/`empty`, `open`/`closed`, `folded`/`unfolded` — the annotator only ever writes one predicate (say `filled`) and negates it when they mean the opposite. The BDDL compiler then applies **De Morgan's law** to push every negation down to atomic formulae, and swaps `not(filled(x))` for `empty(x)` (a *different*, independently-defined predicate, not `filled(x) = False`) before handing the definition to the solver. This matters because a continuously-valued physical quantity (how much liquid is in a container) doesn't have a natural single threshold for "true" vs. "false" — treating the two directions as separate predicates each with their own sampling/checking function (Section 5.1) lets each get its own threshold rather than forcing one boundary to serve both meanings.

**Composition/decomposition — the `future` predicate.** Standard PDDL assumes every object in `:objects` exists for the whole episode. BEHAVIOR-1K activities need transition rules that *create* new objects mid-episode (dough → pie). The fix: any object that doesn't exist in the scene at `t=0` but must exist for the goal to be checkable is declared with a `future` predicate in `:init`, and appears **nowhere else** in `:init` (since it doesn't exist yet, it can't have any other predicates true of it). This is the annotation-time signal that tells the sampler and the transition machine "this object gets instantiated later, by a transition rule, not by initial scene sampling.\"

In [ ]:
from dataclasses import dataclass, field


# The real BDDL three-valued-predicate mechanism: paired predicates whose
# negations get swapped via De Morgan's law rather than left as a bare `not`.
THREE_VALUED_PAIRS = {
    "Filled": "Empty",
    "Open": "Closed",
    "Folded": "Unfolded",
}
THREE_VALUED_PAIRS.update({v: k for k, v in THREE_VALUED_PAIRS.items()})  # symmetric lookup


def apply_de_morgan_switch(predicate_name: str, negated: bool) -> str:
    """Given a predicate an annotator wrote (possibly negated), return the
    predicate name BDDL actually compiles it to -- swapping to the paired
    predicate under negation instead of leaving a bare boolean negation,
    exactly as Section 4.1 describes.
    """
    if not negated:
        return predicate_name
    if predicate_name in THREE_VALUED_PAIRS:
        return THREE_VALUED_PAIRS[predicate_name]
    return f"not({predicate_name})"  # ordinary two-valued predicates negate normally


for pred, negated in [("Filled", False), ("Filled", True), ("Open", True), ("ToggledOn", True)]:
    compiled = apply_de_morgan_switch(pred, negated)
    print(f"annotator wrote {'not ' if negated else ''}{pred:12s} -> BDDL compiles to: {compiled}")

## 5. Theory: OmniGibson — why a new simulator, and what it adds

BEHAVIOR-100 (the predecessor) ran on iGibson 2.0. The paper's finding, stated plainly, is that **iGibson 2.0 cannot realistically simulate BEHAVIOR-1K's activities** — over half of BEHAVIOR-1K's 1,000 activities would be un-simulatable without capabilities iGibson 2.0 lacks. That gap is what motivates OmniGibson, built on **NVIDIA Omniverse and PhysX 5**.

**What OmniGibson adds over a typical rigid-body robotics simulator:**

- **Deformable bodies and cloth** — folding a towel, deforming dough.
- **Fluid simulation** — pouring, soaking, filling — not just "liquid present/absent" but actual fluid dynamics.
- **Extended, non-kinematic object states** — temperature, soaked-level, dirtiness — driven by heuristics (e.g. an object's temperature rises when adjacent to a toggled-on heat source) layered on top of the physics engine, since these aren't things a rigid/soft-body physics solver represents natively.
- **The Transition Machine** (Section 3) — symbolic swap-rules for processes physics alone won't capture (dough $\to$ pie in an oven above a threshold temperature).
- **Photorealistic ray/path-traced rendering** — the paper ran its own visual-realism study (60 human raters scoring sampled images 1–5) and found OmniGibson rated significantly more realistic than Habitat 2.0, AI2-THOR, iGibson 2.0, and ThreeDWorld. This isn't just aesthetics: sim-to-real transfer (Section 9) depends heavily on how close simulated images are to what a real camera sees.
- **Native support for infinite valid initial-condition sampling and goal-condition checking** driven directly by an activity's BDDL predicates — the simulator can generate a fresh, valid scene configuration satisfying an activity's `:init` predicates, and later programmatically check whether the `:goal` predicates hold, without a human in the loop for either.

**The explicit tradeoff, stated by the authors themselves:** all this realism costs speed. OmniGibson runs at roughly **60 fps** for a house scene with ~60 objects, versus roughly 100 fps for the same scene in iGibson 2.0 — worth knowing before you assume you can just brute-force millions of RL environment steps the way you might in a lighter simulator.

### 5.1 Extended object states and logical predicates: the exact specification

This is the mechanism that actually makes BDDL goal-checking and initial-state sampling work, at the level of detail in the paper's Appendix E.1.

**Selective state tracking, for computational efficiency.** OmniGibson does not track every extended state for every object — it looks up which object-category properties (Table A.1 in Section 3) an object has, and only instantiates the states those properties require:

| Object category property | Extended state(s) tracked |
|---|---|
| `cookable`, `freezable`, `flammable`, `heatable`, `meltable` | `Temperature`, `MaxTemperature` |
| `soakable` | `SoakedLevel` (tracked *per liquid type*) |
| `toggleable` | `ToggledState` |
| `sliceable` | `SlicedState` |
| `breakable` | `BrokenState` |
| `heatSource` / `fireSource` / `coldSource` / `waterSource` | `ToggledState` |

An apple needs `Temperature` tracked (it's `cookable`); a table doesn't, because nothing in BEHAVIOR-1K ever checks a table's temperature. This selectivity is exactly what Section 10.1's performance-benchmark table is measuring the cost of when it's turned off.

**Logical predicates are checking functions AND sampling functions — two separate pieces of code per predicate.** A *checking* function maps the current physical state to a Boolean (used for reward/goal evaluation); a *sampling* function does the reverse — given a target Boolean value, it mutates the world to make the predicate true or false (used for initial-state generation). Both are precisely defined per predicate, with numeric thresholds. A representative subset:

| Predicate | Checking function | Sampling function |
|---|---|---|
| `Cooked(o)` | True iff $T^{max}_o$ has reached $T_{cooked}$ (annotated per category) at any point in the episode, and stayed below $T_{burnt}$ | To sample True: set $T^{max}_o \leftarrow \max(T^{max}_o, T_{cooked})$. To sample False: set $T^{max}_o \leftarrow \min(T^{max}_o, T_{cooked}-1)$ |
| `Frozen(o)` | True iff current $T_o \le T_{frozen}$ (default $0°C$) | To sample True: draw $T_o$ uniformly from $[T_{frozen}-50,\, T_{frozen}-10]$. To sample False: set $T_o \leftarrow T_{frozen}+1$ |
| `OnFire(o)` | True iff current $T_o \ge T_{onfire}$ (default $300°C$) | To sample True: draw $T_o$ uniformly from $[T_{onfire}+10,\, T_{onfire}+50]$. To sample False: set $T_o \leftarrow T_{onfire}-1$ |
| `Soaked(o, l)` | True iff `SoakedLevel` $w \ge w_{soaked}$ (default 50 particles) for liquid $l$ | To sample True: set $w \leftarrow w_{soaked}$. To sample False: set $w \leftarrow 0$ |
| `Filled(o, l)` | True iff particle count of $l$ inside `ContainerVolume(o)` exceeds threshold fraction $w_{filled}$ (default 0.5) of the container's volume | To sample True: place enough particles of $l$ inside the container to exceed $w_{filled}$. To sample False: remove all particles of $l$ from the container |
| `Open(o)` | True iff a `RelevantJoints`-annotated joint's position $q$ exceeds 5% of its range: $q > 0.05(q_{upper}-q_{lower}) + q_{lower}$ | To sample True: move a subset of relevant joints to a uniformly random position between the 5% threshold and the upper limit. To sample False: move all relevant joints between the lower limit and the threshold |
| `OnTopOf(o1, o2)` | True iff $o_2$ is in $o_1$'s negative-vertical-axis ray-cast set, NOT in its positive-vertical-axis set, AND the two are in physical contact | Only True can be sampled: ray-cast candidate poses for $o_1$ above $o_2$'s surface until a collision-free, fully-supported pose is found |

Notice the pattern: **every threshold is a number the researchers had to pick and could get wrong** — $w_{soaked}=50$ particles, $w_{filled}=0.5$, $T_{onfire}=300°C$. These are stated as defaults, explicitly configurable per object category and model, which is the right design if you want your object-property annotation (Section 3) to eventually override a bad global default for a specific weird object (e.g. a doll house that shouldn't catch "fire" at real-world temperatures) — but it also means the model of the world every RL baseline in Section 7 is trained against is only as accurate as these hand-set constants.

In [ ]:
# A faithful (not toy) implementation of a subset of OmniGibson's checking
# and sampling functions from Table A.8 / A.9, with the paper's default thresholds.

class ObjectPhysicalState:
    """Minimal per-object physical state -- just enough fields to exercise the
    real checking/sampling functions below.
    """
    def __init__(self, name: str):
        self.name = name
        self.temperature = 20.0        # deg C, room temperature default
        self.max_temperature = 20.0    # historical max, deg C
        self.soaked_level = {}         # {liquid_name: particle_count}
        self.container_fill_frac = {}  # {liquid_name: fraction of ContainerVolume filled}


# --- default thresholds, exactly as specified in Table A.8 ---
T_COOKED_DEFAULT = 70.0     # annotated per-category in the real system; illustrative here
T_BURNT_DEFAULT = 150.0
T_FROZEN = 0.0
T_ONFIRE = 300.0
T_HEATED = 75.0
T_BOILED = 100.0
W_SOAKED = 50           # particle count threshold
W_FILLED = 0.5          # fraction of ContainerVolume


def check_cooked(o: ObjectPhysicalState, t_cooked=T_COOKED_DEFAULT, t_burnt=T_BURNT_DEFAULT) -> bool:
    return t_cooked <= o.max_temperature < t_burnt

def sample_cooked(o: ObjectPhysicalState, value: bool, t_cooked=T_COOKED_DEFAULT):
    o.max_temperature = max(o.max_temperature, t_cooked) if value else min(o.max_temperature, t_cooked - 1)

def check_frozen(o: ObjectPhysicalState, t_frozen=T_FROZEN) -> bool:
    return o.temperature <= t_frozen

def sample_frozen(o: ObjectPhysicalState, value: bool, rng: np.random.Generator, t_frozen=T_FROZEN):
    o.temperature = rng.uniform(t_frozen - 50, t_frozen - 10) if value else t_frozen + 1

def check_onfire(o: ObjectPhysicalState, t_onfire=T_ONFIRE) -> bool:
    return o.temperature >= t_onfire

def check_soaked(o: ObjectPhysicalState, liquid: str, w_soaked=W_SOAKED) -> bool:
    return o.soaked_level.get(liquid, 0) >= w_soaked

def sample_soaked(o: ObjectPhysicalState, liquid: str, value: bool, w_soaked=W_SOAKED):
    o.soaked_level[liquid] = w_soaked if value else 0

def check_filled(o: ObjectPhysicalState, liquid: str, w_filled=W_FILLED) -> bool:
    return o.container_fill_frac.get(liquid, 0.0) >= w_filled


# --- exercise the real mechanism on a toy "cloth" and "mug" ---
rng = np.random.default_rng(SEED)
cloth = ObjectPhysicalState("cloth_1")
mug = ObjectPhysicalState("mug_1")

print("Before sampling:")
print(f"  Frozen(cloth) = {check_frozen(cloth)}, Cooked(mug's contents) = {check_cooked(mug)}")

sample_frozen(cloth, True, rng)
sample_soaked(cloth, "water", True)
mug.container_fill_frac["coffee"] = 0.0

print("\nAfter sampling Frozen(cloth)=True and Soaked(cloth, water)=True:")
print(f"  cloth.temperature = {cloth.temperature:.1f}C  ->  Frozen(cloth) = {check_frozen(cloth)}")
print(f"  cloth.soaked_level[water] = {cloth.soaked_level['water']}  ->  Soaked(cloth, water) = {check_soaked(cloth, 'water')}")
print(f"  Filled(mug, coffee) = {check_filled(mug, 'coffee')}  (fill fraction {mug.container_fill_frac['coffee']} < {W_FILLED})")

## 6. Theory: how BEHAVIOR-1K compares to other embodied AI benchmarks

The paper's own comparison table (condensed and paraphrased below) is worth internalizing because it names the specific axes along which embodied-AI benchmarks trade off against each other — the same axes you should be checking whenever you evaluate a new benchmark for your own project:

| Benchmark | # activities | Scene types | Object categories | Object models | Diversity source | Physical realism |
|---|---|---|---|---|---|---|
| **BEHAVIOR-1K** | 1,000 | 8 (houses, offices, restaurants, stores, ...) | 1,949 | 9,318 | human-preference survey | rigid + deformable + fluid + thermal |
| BEHAVIOR-100 | 100 | 1 (houses) | 391 | 1,217 | researcher-selected (subset of same survey lineage) | rigid + limited extended states |
| Habitat 2.0 (HAB) | 1 (rearrangement) | 1 | ~117 | ~162 | researcher-selected | rigid body only |
| AI2-THOR / ALFRED | ~2–7 task families | 1 | ~84–118 | ~84–118 | researcher-selected | limited physical realism |
| RLBench | 100 | 1 (tabletop) | ~28 | ~28 | researcher-selected | rigid body only |
| SoftGym | 5 | 1 | — | — | researcher-selected | deformable-focused, narrow task set |

**The pattern the paper explicitly names: a diversity-realism tradeoff exists across almost every prior benchmark.** Instruction-following benchmarks (VirtualHome, ALFRED) are diverse in scenes/objects/states but simulate physics loosely. Rearrangement benchmarks with careful physics (Habitat 2.0, SAPIEN ManiSkill) support few tasks. Deformable-focused simulators (SoftGym, RFUniverse) have realism close to OmniGibson's but a narrow task set. BEHAVIOR-1K's claimed contribution is specifically to **not** make this tradeoff — high diversity (1,000 activities, 50 scenes, 1,949 object categories) *and* high physical realism (fluids, deformables, thermal effects) simultaneously, at real engineering cost (the 60fps ceiling noted above, and the installation weight covered in Section 10).

If you're asked to place a new embodied-AI paper in context, this diversity-vs-realism table is the fastest way to do it: find where the new benchmark sits on both axes relative to this list, rather than evaluating it in isolation.

**Visualizing the diversity-realism tradeoff** — plotting each benchmark from the table above on two axes makes the tradeoff (and BEHAVIOR-1K's claimed escape from it) visually obvious in a way the table alone doesn't:

In [ ]:
# axis values are relative/illustrative rankings derived from the table above
# (activity count on a log scale as a diversity proxy; a 1-5 realism score
# reflecting rigid-only vs. rigid+deformable vs. +fluid+thermal support)
benchmarks = {
    "BEHAVIOR-1K":   (1000, 5.0),
    "BEHAVIOR-100":  (100,  3.5),
    "Habitat 2.0":   (1,    2.5),
    "AI2-THOR/ALFRED": (5,  1.5),
    "RLBench":       (100,  2.0),
    "SoftGym":       (5,    3.0),
}

fig, ax = plt.subplots(figsize=(7.5, 5.5))
for name, (diversity, realism) in benchmarks.items():
    color = "#ef5350" if name == "BEHAVIOR-1K" else "#4fc3f7"
    size = 220 if name == "BEHAVIOR-1K" else 120
    ax.scatter(diversity, realism, s=size, color=color, edgecolor="white", linewidth=0.8, zorder=3)
    ax.annotate(name, (diversity, realism), textcoords="offset points", xytext=(8, 6), fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("task/activity diversity (# activities, log scale)")
ax.set_ylabel("physical realism (rigid → +deformable → +fluid/thermal)")
ax.set_title("The diversity-realism tradeoff most prior benchmarks accept")
ax.set_ylim(1, 5.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Theory: the three evaluated baselines

The paper doesn't attempt to solve all 1,000 activities — it picks **three paradigmatic activities** that each stress a different manipulation capability, and evaluates three RL baselines on them:

**The three activities:**
- **CollectTrash** — gather empty bottles/cups and throw them in a bin. Stresses rigid-body manipulation *and* long-horizon multi-object search (at least 16 primitive steps for an optimal solution).
- **StoreDecoration** — store items into a drawer. Stresses articulated-object manipulation (opening/closing the drawer as part of the task).
- **CleanTable** — wipe a dirty table with a soaked cloth. Stresses flexible-material and fluid manipulation (the cloth must actually be wet; wiping must actually remove dirt particles), solvable in as few as 6 optimal primitive steps.

**The three baselines**, all trained with a **sparse task-success reward only** (no reward shaping):

| Baseline | RL algorithm | Action space | Memory |
|---|---|---|---|
| **RL-VMC** | Soft Actor-Critic (SAC) | raw visuomotor control: image $\to$ low-level joint commands | none |
| **RL-Prim.** | PPO | discrete choice of **action primitive** (pick, place, push, navigate, dip, wipe) + target object, executed via sampling-based motion planning | none |
| **RL-Prim.Hist.** | PPO | same as RL-Prim. | last 3 observation steps, to disambiguate visually aliased states |

**Why action primitives at all, and what they simplify.** A primitive like "pick object X" isn't a raw joint-torque policy — the paper's implementation checks only whether the *final* configuration (e.g. the grasp pose) is kinematically feasible (reachable, collision-free) and, if so, teleports the robot state directly there rather than simulating the full approach trajectory during training. This is a deliberate, explicitly-flagged simplification to make training tractable at all — and Section 8 covers what happens when you remove it at evaluation time.

### 7.1 The exact reward function and RL objectives

**Reward function.** Every baseline uses the *same* reward: the BDDL goal-satisfaction signal itself, with no shaping. Concretely, for `StoreDecoration`, the BDDL goal is (in the paper's notation) `Forall(decoration){Inside(decoration, cabinet)}` — the agent receives $r_t = 1$ at the timestep this becomes true, and $r_t = 0$ at every other timestep, including every intermediate sub-step (e.g. the drawer must be pushed open *before* a decoration can be placed inside it, but opening the drawer itself earns no reward). This is what "no reward engineering" means concretely, and it's the direct cause of Finding 1 (RL-VMC's total failure) in Section 8 — a policy exploring randomly under a purely binary, multi-step-delayed reward has essentially no gradient signal to climb until it stumbles into the full completed sequence by chance.

**RL-Prim. / RL-Prim.Hist. — PPO's clipped surrogate objective:**

$$
L^{CLIP}(\theta) = \hat{\mathbb{E}}_t\Big[\min\big(r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta),\, 1-\epsilon,\, 1+\epsilon)\,\hat{A}_t\big)\Big]
$$

where $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the probability ratio between the new and old policy, $\hat{A}_t$ is the estimated advantage, and $\epsilon$ caps how far a single update can push the policy ratio away from 1 — the mechanism that keeps PPO's on-policy updates stable.

**RL-VMC — SAC's maximum-entropy objective:**

$$
\pi^{*} = \arg\max_{\pi}\ \mathbb{E}_{\tau\sim\pi}\left[\sum_{t=0}^{\infty}\gamma^t\big(R(s_t,a_t,s_{t+1}) + \alpha\, \mathcal{H}(\pi(\cdot|s_t))\big)\right]
$$

where $\alpha$ weights the entropy bonus $\mathcal{H}$ against the task reward — SAC explicitly rewards *staying stochastic*, which is meant to aid exploration, but per Finding 1 in Section 8, that exploration bonus alone isn't enough to overcome the sparse, long-horizon reward at the raw joint-command action space's granularity.

**Exact hyperparameters** (both trained for 30,000 timesteps, 3 seeds, evaluated on seed 0):

| SAC (RL-VMC) | Value |
|---|---|
| Learning rate | 0.0003 |
| Buffer size | 300 |
| Batch size | 64 |
| Discount $\gamma$ | 0.99 |
| Soft update coefficient $\tau$ | 0.005 |

| PPO (RL-Prim. / RL-Prim.Hist.) | Value |
|---|---|
| Learning rate | 0.0003 |
| Buffer size | 300 |
| Batch size | 64 |
| Discount $\gamma$ | 0.99 |
| GAE parameter $\gamma_{gae}$ | 0.99 |
| Clipping parameter $\epsilon$ | 0.2 |
| Entropy coefficient $c_1$ | 0.0 |
| Value-function coefficient $c_2$ | 0.5 |

**The detail worth flagging in a meeting:** PPO's entropy coefficient here is **0.0** — no explicit entropy bonus at all — while SAC's *entire* objective is built around maximizing entropy. That's a genuine asymmetry in how much these two algorithms are pushed to explore, on top of the difference in action space (discrete primitive choice vs. continuous joint commands), so Finding 1's "RL-VMC fails, RL-Prim. doesn't" can't be attributed to the action-space abstraction alone without also accounting for this.

In [ ]:
# Numerically verifying both objectives on toy data -- not a training loop,
# just confirming you can compute what the equations above actually specify.

def ppo_clipped_surrogate(old_log_probs, new_log_probs, advantages, epsilon=0.2):
    """L^CLIP(theta) from Eq. 1 -- the exact PPO objective RL-Prim./RL-Prim.Hist. optimize."""
    ratio = np.exp(new_log_probs - old_log_probs)          # r_t(theta)
    unclipped = ratio * advantages
    clipped = np.clip(ratio, 1 - epsilon, 1 + epsilon) * advantages
    return np.mean(np.minimum(unclipped, clipped))


def sac_max_entropy_return(rewards, entropies, gamma=0.99, alpha=0.2):
    """The discounted (reward + alpha * entropy) return from Eq. 2 -- what SAC's
    RL-VMC baseline maximizes over a single sampled trajectory.
    """
    discounts = gamma ** np.arange(len(rewards))
    return np.sum(discounts * (rewards + alpha * entropies))


rng = np.random.default_rng(SEED)

# PPO: a batch of toy (old_log_prob, new_log_prob, advantage) triples
old_lp = rng.normal(-1.0, 0.3, size=64)
new_lp = old_lp + rng.normal(0.0, 0.15, size=64)   # policy has moved a bit since the rollout
advantages = rng.normal(0.0, 1.0, size=64)
objective = ppo_clipped_surrogate(old_lp, new_lp, advantages, epsilon=0.2)
print(f"PPO clipped surrogate objective (toy batch, epsilon=0.2): {objective:.4f}")

# show the clipping actually bites for a large policy shift
old_lp_big_shift = rng.normal(-1.0, 0.3, size=64)
new_lp_big_shift = old_lp_big_shift + rng.normal(0.0, 1.5, size=64)  # much bigger update
objective_unclipped_equivalent = np.mean(np.exp(new_lp_big_shift - old_lp_big_shift) * advantages)
objective_clipped = ppo_clipped_surrogate(old_lp_big_shift, new_lp_big_shift, advantages, epsilon=0.2)
print(f"Large policy shift -- naive (unclipped) objective: {objective_unclipped_equivalent:.4f}, "
      f"PPO's actual clipped objective: {objective_clipped:.4f}")
print("(clipping caps how much a single large ratio can move the objective, which is exactly")
print(" the update-stability mechanism the epsilon=0.2 hyperparameter above controls)")

# SAC: a toy sparse-reward trajectory (reward=1 only at the final successful step)
horizon = 20
rewards = np.zeros(horizon); rewards[-1] = 1.0
entropies = rng.uniform(0.5, 1.5, size=horizon)  # policy entropy per timestep
ret = sac_max_entropy_return(rewards, entropies, gamma=0.99, alpha=0.2)
print(f"\nSAC max-entropy discounted return (sparse reward, alpha=0.2): {ret:.4f}")
print("Most of this return comes from the entropy bonus, not the task reward --")
print("exactly the exploration-vs-exploitation balance the sparse-reward setting stresses.")

### 7.2 Network architecture (Fig. A.11, exact spec)

**RL-Prim. / RL-Prim.Hist. (PPO):** a shared visual feature extractor plus a small state encoder, fused and split into value/action heads.
- Visual input: $128\times128\times3$ egocentric RGB, per-channel normalized with a moving average.
- Visual feature extractor: **Conv → ReLU → MaxPool → Flatten** (a single conv-pool block, not a deep ResNet — this is a deliberately lightweight architecture given the 30,000-timestep training budget).
- State encoder: a small MLP taking a single input — whether the robot is currently grasping an object (a Boolean/scalar).
- Fusion: visual features + state encoding → MLP → a 128-dimensional joint representation.
- Two heads off that 128-dim representation: a **value head** (scalar, for PPO's advantage estimation) and a **discrete action head** (selects an action primitive + target object).

**RL-VMC (SAC):** the same visual feature extractor, feeding into three separate MLPs — actor, critic, and target critic (SAC's usual moving-average target network for stability) — all with ReLU activations. Unlike RL-Prim., RL-VMC's action space is **continuous**: the actor outputs low-level joint commands directly, with no discrete primitive layer at all.

The architecture is intentionally small on both sides — the paper's point (Section 8) is that even this modest network, given primitives, gets non-trivial success, while the same visual backbone feeding a continuous low-level action space gets exactly zero. The bottleneck in Finding 1 is the credit-assignment problem from the action space and reward sparsity (Section 7.1), not network capacity.

In [ ]:
import torch
import torch.nn as nn


class VisualFeatureExtractor(nn.Module):
    """Conv-ReLU-MaxPool-Flatten, exactly as specified in Fig. A.11 -- a single
    lightweight conv-pool block, not a deep backbone, operating on 128x128x3 images.
    """
    def __init__(self, out_dim: int = 128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 128, 128)
            flat_dim = self.conv(dummy).flatten(1).shape[1]
        self.proj = nn.Linear(flat_dim, out_dim)

    def forward(self, images):
        return self.proj(self.conv(images).flatten(1))


class RLPrimPolicy(nn.Module):
    """RL-Prim. / RL-Prim.Hist.'s PPO network: visual features + grasp-state encoding,
    fused into a 128-dim representation, split into a value head and a discrete
    action-primitive head. `history_len` > 1 reproduces RL-Prim.Hist.'s use of
    3 past observation steps.
    """
    def __init__(self, n_primitive_actions: int = 6, history_len: int = 1):
        super().__init__()
        self.history_len = history_len
        self.visual = VisualFeatureExtractor(out_dim=128)
        self.state_encoder = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 128))
        self.fusion = nn.Sequential(nn.Linear(128 * history_len * 2, 256), nn.ReLU(), nn.Linear(256, 128))
        self.value_head = nn.Linear(128, 1)
        self.action_head = nn.Linear(128, n_primitive_actions)

    def forward(self, image_history, is_grasping_history):
        # image_history: (B, history_len, 3, 128, 128); is_grasping_history: (B, history_len, 1)
        B, T = image_history.shape[:2]
        vis_feats = self.visual(image_history.reshape(B * T, 3, 128, 128)).view(B, T * 128)
        state_feats = self.state_encoder(is_grasping_history.reshape(B * T, 1)).view(B, T * 128)
        fused = self.fusion(torch.cat([vis_feats, state_feats], dim=-1))
        return self.value_head(fused), self.action_head(fused)  # (value, action logits)


class RLVMCNetworks(nn.Module):
    """RL-VMC's SAC networks: shared visual backbone, three separate MLPs
    (actor, critic, target critic), continuous low-level action output --
    no discrete primitive layer anywhere.
    """
    def __init__(self, action_dim: int = 7):
        super().__init__()
        self.visual = VisualFeatureExtractor(out_dim=128)
        self.actor = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, action_dim))
        self.critic = nn.Sequential(nn.Linear(128 + action_dim, 128), nn.ReLU(), nn.Linear(128, 1))
        self.target_critic = nn.Sequential(nn.Linear(128 + action_dim, 128), nn.ReLU(), nn.Linear(128, 1))

    def forward(self, image):
        feats = self.visual(image)
        action = self.actor(feats)
        q = self.critic(torch.cat([feats, action], dim=-1))
        return action, q


# sanity-check both architectures on random data
rl_prim = RLPrimPolicy(n_primitive_actions=6, history_len=3)
imgs = torch.randn(2, 3, 3, 128, 128)
grasp_hist = torch.randint(0, 2, (2, 3, 1)).float()
value, action_logits = rl_prim(imgs, grasp_hist)
print("RL-Prim.Hist. -- value shape:", tuple(value.shape), " action logits shape:", tuple(action_logits.shape))

rl_vmc = RLVMCNetworks(action_dim=7)
img_single = torch.randn(2, 3, 128, 128)
action, q_value = rl_vmc(img_single)
print("RL-VMC        -- continuous action shape:", tuple(action.shape), " Q-value shape:", tuple(q_value.shape))

## 8. Headline experimental results, with the caveats attached

**Finding 1 — end-to-end visuomotor RL fails completely.** RL-VMC achieves **0% success on all three activities**. The paper attributes this to the combination of long horizon, sparse reward, and raw joint-level action space — the credit-assignment and exploration problems compound badly enough that SAC never finds a working policy at all, not even a mediocre one.

**Finding 2 — action primitives make the problem tractable, but not easy.** RL-Prim. reaches 42–77% success depending on the activity (lowest on CollectTrash, highest on CleanTable), and RL-Prim.Hist. improves further to 55–88%. The paper's reading: **some form of temporal/action-space abstraction is close to necessary** for long-horizon BEHAVIOR-1K activities with current RL methods — this echoes a repeated finding across the embodied-AI literature (also true of LIBERO's discussion of hierarchical methods like BUDS/LOTUS, if you've read that notebook).

**Finding 3 — memory matters more as horizon grows.** The RL-Prim. $\to$ RL-Prim.Hist. gain is largest on **CollectTrash** (the longest-horizon activity, 16+ primitive steps), because without any memory of which locations have already been checked/cleaned, the policy repeatedly re-visits already-handled objects — a visually-aliased-state problem that a 3-step observation history substantially mitigates.

**Finding 4 — grasping realism is the single biggest hidden assumption.** An ablation removing the "assistive grasp" simplification (i.e. forcing genuinely physics-based grasping instead of a rigid snap-to-gripper connection) causes a **radical performance drop** — success collapses toward zero across all three activities. In sharp contrast, removing the *motion-execution* simplification (full trajectory simulation instead of teleporting to the planned endpoint) causes comparatively little performance loss. **The takeaway that matters for anyone building on this benchmark:** if your baseline's action primitives assume simplified grasping, your reported numbers are measuring something meaningfully easier than the activity as actually specified — grasping, not motion planning, is where the difficulty is hiding.

If you're presenting this paper, Findings 1 and 4 are the two most likely to get "wait, so nothing actually works?" pushback — worth having the framing ready that this is precisely the paper's point: BEHAVIOR-1K activities are *designed* to be beyond current methods' reach, as a target for the field rather than a benchmark current methods are expected to already solve well.

**Visualizing Table 2 (task success rates)** — the three baselines across the three activities, as a grouped bar chart:

In [ ]:
activities = ["StoreDecoration", "CollectTrash", "CleanTable"]
methods = {
    "RL-VMC":        [0.00, 0.00, 0.00],
    "RL-Prim.":      [0.48, 0.42, 0.77],
    "RL-Prim.Hist.": [0.55, 0.63, 0.88],
}
method_colors = {"RL-VMC": "#ef5350", "RL-Prim.": "#ffca28", "RL-Prim.Hist.": "#66bb6a"}

x = np.arange(len(activities))
width = 0.25

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for i, (method, rates) in enumerate(methods.items()):
    ax.bar(x + (i - 1) * width, rates, width, label=method, color=method_colors[method])

ax.set_xticks(x)
ax.set_xticklabels(activities)
ax.set_ylabel("task success rate")
ax.set_ylim(0, 1.0)
ax.set_title("Table 2 reproduced: task success rate by method and activity")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Note the flat zero row for RL-VMC across every activity -- Finding 1 from")
print("Section 8 made visually unambiguous: raw visuomotor RL doesn't get partial")
print("credit anywhere, it fails completely, everywhere.")

**Visualizing Table 4 (the grasping/motion ablation)** — this is the single most important chart in the notebook for understanding where BEHAVIOR-1K's real difficulty hides:

In [ ]:
ablation_conditions = ["Simplified grasp\n+ teleport motion\n(training setup)",
                        "Full physics grasp\n+ teleport motion",
                        "Simplified grasp\n+ full motion sim"]
ablation_rates = {
    "StoreDecoration": [0.48, 0.00, 0.46],
    "CollectTrash":    [0.42, 0.00, 0.36],
    "CleanTable":      [0.77, 0.00, 0.73],
}

x = np.arange(len(ablation_conditions))
width = 0.25
fig, ax = plt.subplots(figsize=(9.5, 4.5))
colors = ["#4fc3f7", "#ffca28", "#ef5350"]
for i, (activity, rates) in enumerate(ablation_rates.items()):
    ax.bar(x + (i - 1) * width, rates, width, label=activity, color=colors[i])

ax.set_xticks(x)
ax.set_xticklabels(ablation_conditions, fontsize=9)
ax.set_ylabel("task success rate")
ax.set_ylim(0, 1.0)
ax.set_title("Table 4 reproduced: removing simplifications at evaluation time")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Full physics grasping (middle group) collapses every activity to 0% --")
print("while full trajectory motion simulation (right group) barely moves the")
print("numbers at all versus the original training setup (left group). This is")
print("the chart to point to when someone asks 'so what's actually hard here?'")

## 9. Theory: the sim-to-real study

To calibrate how much OmniGibson's realism actually closes the sim-to-real gap, the authors scanned a real mockup apartment into a matching virtual scene (a "digital twin"), and ran **CollectTrash** on both a simulated and a real bimanual mobile manipulator (**Tiago**), using RGB-D + YOLOv3 object detection and particle-filter LiDAR localization on the real robot.

**Success rates:** ~40% in simulation (50 runs) vs. ~22% real-world with an *optimal* (human-scripted) primitive-selection policy (27 runs) vs. **0%** real-world with the policy actually *trained* in OmniGibson (26 runs).

**Where the gap comes from (failure attribution):**
- In simulation, most failures are attributed to the *visual policy* itself (choosing the wrong primitive) plus stochasticity in the place primitive and motion planner — recall grasping failures don't show up in sim because of the assistive-grasp simplification from Section 7.
- On the real robot, **grasping** becomes a major failure source for both policies (~40% of failures) — direct empirical confirmation of Finding 4 in Section 8: once you can't cheat grasping, it's genuinely hard.
- For the *trained* policy specifically, ~44% of real-world errors trace to the visual policy picking the wrong primitive because of a **visual domain gap** — unmodeled effects like the real camera's limited dynamic range and imperfect surface-texture/reflectivity modeling in the 3D assets.
- A subtler, compounding failure mode: **navigation inaccuracy in a previous timestep leaves the robot base badly positioned for the next manipulation step** — an error that doesn't exist in simulation at all, because the simulated setup assumes perfect localization and execution. This is a good example of a sim-to-real gap that isn't about visual fidelity or physics accuracy at all, but about an assumption (perfect localization) baked silently into the simulated evaluation protocol.

**Practical implication if you're planning your own sim-to-real project on this benchmark:** don't only invest in closing the *visual* domain gap (textures, lighting, camera modeling) — the paper's own failure breakdown suggests grasping realism and compounding localization error are comparably large contributors, and neither is a rendering problem.

**Visualizing the sim-to-real gap** — success rate collapse across the three evaluated conditions:

In [ ]:
conditions = ["Simulation\n(50 runs)", "Real, optimal\npolicy (27 runs)", "Real, trained\npolicy (26 runs)"]
sim2real_rates = [0.40, 0.22, 0.00]

fig, ax = plt.subplots(figsize=(7, 4.3))
bars = ax.bar(conditions, sim2real_rates, color=["#66bb6a", "#ffca28", "#ef5350"])
ax.set_ylabel("CollectTrash success rate")
ax.set_ylim(0, 0.5)
ax.set_title("The sim-to-real gap on CollectTrash")
for bar, rate in zip(bars, sim2real_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{rate:.0%}",
            ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Illustrative failure-attribution breakdown, paraphrased from the paper's
# qualitative description (Fig. 5): grasping and perception dominate real-world
# failures, while simulation failures are almost entirely policy/planner related.
failure_categories = ["Grasping", "Perception\n(wrong primitive)", "Policy/planner\nstochasticity"]
sim_failures =        [0,  15, 85]   # % of failures, simulation (no grasping failures -- assistive grasp)
real_trained_failures = [40, 44, 16]  # % of failures, real world, trained policy
real_optimal_failures = [45, 0,  55]  # % of failures, real world, optimal (scripted) policy

x = np.arange(len(failure_categories))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - width, sim_failures, width, label="Simulation", color="#4fc3f7")
ax.bar(x, real_trained_failures, width, label="Real, trained policy", color="#ef5350")
ax.bar(x + width, real_optimal_failures, width, label="Real, optimal policy", color="#ffca28")
ax.set_xticks(x)
ax.set_xticklabels(failure_categories)
ax.set_ylabel("% of failures attributed to this cause")
ax.set_title("Where failures come from: sim vs. real (illustrative, from Fig. 5's description)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Grasping failures are literally impossible in simulation here (0%) because")
print("of the assistive-grasp simplification -- they only appear once you're on")
print("real hardware. That gap between the blue bar and the other two, at the")
print("'Grasping' category, is the sim-to-real gap made visible in one chart.")

## 10. Installation — and an important scope-setting note before you try

**This is a fundamentally heavier stack than LIBERO or CALVIN.** OmniGibson is built on **NVIDIA Omniverse and Isaac Sim**, which requires:

- An **NVIDIA RTX-capable GPU** (Omniverse's PhysX 5 + ray/path-traced rendering are not CPU-simulatable, unlike MuJoCo/`robosuite` in the LIBERO notebook) — there is no CPU-only fallback for actually running the simulator.
- A specific, fairly heavy install (Isaac Sim + Omniverse Kit + OmniGibson + BDDL), documented at the [official installation guide](https://behavior.stanford.edu/omnigibson/getting_started/installation.html), typically tens of GB.
- Terms-of-service acceptance for conda, the NVIDIA EULA, and the dataset license — the install script supports non-interactive flags for automated/CI setups, shown below.

**Practical consequence for this notebook, and for `xulabs/edu` conventions generally:** unlike the LIBERO tutorial, this notebook **cannot be made CPU-friendly or reliably Colab-runnable** without a paid/GPU-backed runtime and a substantial one-time setup — that's a real constraint worth flagging explicitly before this goes into a PR, rather than writing install cells that quietly won't work for most readers. The official repo does provide a Colab image and Docker images if a GPU Colab runtime is available; consider linking those rather than reproducing the install here.

```bash
# Reference only — requires an RTX GPU and is NOT expected to run in a standard/free Colab CPU runtime.
git clone -b v3.7.1 https://github.com/StanfordVL/BEHAVIOR-1K.git
cd BEHAVIOR-1K
./setup.sh --new-env --omnigibson --bddl --joylo --dataset \
            --accept-conda-tos --accept-nvidia-eula --accept-dataset-tos
```

Because of this, **every code cell from here on is written to run standalone in pure Python/NumPy**, illustrating the *concepts* (synset hierarchies, BDDL-style predicate checking, action primitives, the survey statistic) without depending on a live OmniGibson install — genuinely runnable in this notebook as-is, rather than gated behind an `if OMNIGIBSON_AVAILABLE` flag that would realistically almost always be `False` for a reader without dedicated GPU infrastructure.

### 10.1 Performance benchmarking (Table A.10, exact)

The "60fps for a ~60-object house scene" figure from Section 5 has a real ablation behind it. The paper benchmarked simulation steps-per-second (SPS) in two scenes — `Rs_int` (81 objects) and `house_single_floor` (621 objects) — on a single RTX 3080, progressively disabling features:

| Evaluation condition | `Rs_int` (81 objs) SPS | `house_single_floor` (621 objs) SPS |
|---|---|---|
| Full feature set (fluid, cloth, object states, robot) | 24 | 11 |
| − fluid and cloth | 58 | 26 |
| − object state update (also drops fluid/cloth) | 77 | 55 |
| − robot entirely (kinematics-only scene) | 90 | 60 |

**Reading this table correctly:** each row removes an *additional* feature on top of the previous row's removals (it's a nested ablation, not four independent conditions), so the 24→90 SPS range on `Rs_int` is the full cost of everything OmniGibson adds on top of a bare kinematic scene with no robot. Fluid/cloth alone costs roughly 2.4x throughput (24→58); object-state tracking costs a further ~1.3x (58→77); the robot itself costs the smallest additional increment (77→90). If your own project doesn't need fluids or cloth for its activities, Section 5.1's selective-state-tracking mechanism and this table together tell you exactly which OmniGibson features to disable for a real, measured speedup, rather than guessing.

In [ ]:
conditions = ["Full feature set", "− fluid & cloth", "− object state\nupdate", "− robot"]
rs_int_sps = [24, 58, 77, 90]
house_sps = [11, 26, 55, 60]

x = np.arange(len(conditions))
width = 0.35
fig, ax = plt.subplots(figsize=(8.5, 4.3))
ax.bar(x - width/2, rs_int_sps, width, label="Rs_int (81 objects)", color="#4fc3f7")
ax.bar(x + width/2, house_sps, width, label="house_single_floor (621 objects)", color="#ef5350")
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=9)
ax.set_ylabel("simulation steps per second (higher = faster)")
ax.set_title("Table A.10 reproduced: what each OmniGibson feature costs")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice the larger scene (621 objects) pays a proportionally BIGGER fluid/cloth")
print("penalty (11 -> 26, more than 2x) than the smaller scene does (24 -> 58, ~2.4x) --")
print("fluid/cloth simulation cost scales with scene content, not just camera/robot count.")

## 11. Code: exploring the WordNet-derived synset/property hierarchy

Section 3 described how object properties are annotated only at **leaf** synsets and then propagated upward as an **intersection** over descendants. Let's implement that propagation rule directly — it's a small, self-contained piece of logic, and building it yourself is the fastest way to internalize *why* intersection (rather than union) is the correct propagation direction, and what breaks if you get it backwards.

In [ ]:
class Synset:
    """A minimal stand-in for a WordNet-derived synset node in the BEHAVIOR-1K
    object hierarchy. Leaf synsets carry directly-annotated properties; internal
    (non-leaf) synsets derive their properties as the intersection of their
    children's properties, so that referencing a general category in an activity
    definition never silently promises a property some concrete instance lacks.
    """

    def __init__(self, name: str, properties: set | None = None, children: list | None = None):
        self.name = name
        self._own_properties = properties or set()
        self.children = children or []

    @property
    def is_leaf(self) -> bool:
        return len(self.children) == 0

    def properties(self) -> set:
        if self.is_leaf:
            return self._own_properties
        child_props = [child.properties() for child in self.children]
        return set.intersection(*child_props) if child_props else set()


# Build a small illustrative hierarchy: container.n.01 -> {bowl, cloth_bag, bucket}
bowl = Synset("bowl.n.01", {"fillable", "rigidBody"})
cloth_bag = Synset("cloth_bag.n.01", {"cloth", "foldable"})       # NOT fillable with liquid!
bucket = Synset("bucket.n.01", {"fillable", "rigidBody", "openable"})

container = Synset("container.n.01", children=[bowl, cloth_bag, bucket])

print("bowl properties      :", bowl.properties())
print("cloth_bag properties :", cloth_bag.properties())
print("bucket properties    :", bucket.properties())
print("container properties (intersection of children):", container.properties())
print()
print("Notice 'fillable' is NOT in container's derived properties, because cloth_bag")
print("(a valid descendant) isn't fillable with a liquid. This is exactly the")
print("unsolvability the paper describes avoiding: if an activity definition said")
print('\'container.n.01\' must be \'Filled\' with a liquid, and the sampler happened to')
print("instantiate a cloth_bag for it, the activity would be impossible to complete.")
print("Because 'fillable' correctly drops out of the intersection, the activity")
print("author is forced to either specify bowl/bucket directly, or accept that")
print("'container' alone isn't specific enough for a liquid-filling goal.")

In [ ]:
def draw_synset_tree(root: "Synset", ax=None):
    """Draw the synset hierarchy as a simple node-and-edge tree, color-coding
    each node by whether it has the 'fillable' property -- a direct visual
    complement to the printed intersection-propagation output above.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))

    def has_fillable(node):
        return "fillable" in node.properties()

    root_x, root_y = 0.5, 0.85
    ax.scatter([root_x], [root_y], s=1800, color="#66bb6a" if has_fillable(root) else "#616161",
               edgecolor="white", zorder=3)
    ax.text(root_x, root_y, root.name.split(".")[0], ha="center", va="center", fontsize=9, zorder=4)

    n = len(root.children)
    xs = np.linspace(0.15, 0.85, n)
    for x, child in zip(xs, root.children):
        color = "#66bb6a" if has_fillable(child) else "#616161"
        ax.plot([root_x, x], [root_y, 0.25], color="#888888", zorder=1, linewidth=1.5)
        ax.scatter([x], [0.25], s=1400, color=color, edgecolor="white", zorder=3)
        label = child.name.split(".")[0] + "\n{" + ", ".join(sorted(child.properties())) + "}"
        ax.text(x, 0.25, label, ha="center", va="center", fontsize=7.5, zorder=4)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title("green = has 'fillable' property   |   gray = does not", fontsize=9)


fig, ax = plt.subplots(figsize=(8, 4.5))
draw_synset_tree(container, ax=ax)
plt.tight_layout()
plt.show()

## 12. Code: writing and validating a toy BDDL-style activity definition

The real BDDL syntax is a Lisp-like predicate-logic file (as in the LIBERO notebook). Here we build a **small Python object model** capturing the same structure — objects, regions, init predicates, goal predicates — plus the **three-valued predicate** extension from Section 4 (`True` / `False` / `"unknown"`), and a toy checker that validates a proposed world state against an activity's goal. This is the mechanism OmniGibson's real goal-checker implements at much larger scale.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Predicate:
    """A single BDDL-style predicate, e.g. Cooked(apple_1) or OnTop(apple_1, plate_1).
    `value` is True, False, or "unknown" (BEHAVIOR-1K's three-valued extension —
    "unknown" means this activity's definition doesn't constrain this predicate).
    """
    name: str
    args: tuple
    value: object = True  # True / False / "unknown"

    def __repr__(self):
        arg_str = ", ".join(self.args)
        tag = "" if self.value is True else f"={self.value}"
        return f"{self.name}({arg_str}){tag}"


@dataclass
class ActivityDefinition:
    """Toy stand-in for a .bddl activity file: an object list plus init/goal predicates."""
    name: str
    objects: dict            # {instance_name: synset_name}
    init_predicates: list = field(default_factory=list)
    goal_predicates: list = field(default_factory=list)

    def check_goal(self, world_state: dict) -> tuple[bool, list]:
        """Check whether `world_state` (a dict of {(predicate_name, args): bool})
        satisfies every non-"unknown" goal predicate. Returns (success, unmet_predicates).
        """
        unmet = []
        for pred in self.goal_predicates:
            if pred.value == "unknown":
                continue  # unconstrained by this activity's definition -- always fine
            key = (pred.name, pred.args)
            actual = world_state.get(key, False)
            if actual != pred.value:
                unmet.append(pred)
        return (len(unmet) == 0, unmet)


# --- a toy "CleanTable" activity, echoing Section 7's real evaluation activity ---
clean_table = ActivityDefinition(
    name="CleanTable",
    objects={"cloth_1": "rag.n.01", "table_1": "table.n.02", "water_1": "water.n.06"},
    init_predicates=[
        Predicate("Dusty", ("table_1",), True),
        Predicate("Soaked", ("cloth_1",), False),
    ],
    goal_predicates=[
        Predicate("Dusty", ("table_1",), False),      # table must no longer be dusty
        Predicate("Soaked", ("cloth_1",), "unknown"),  # cloth's final wetness isn't constrained
    ],
)

# world state after a (hypothetical) successful rollout
world_after_success = {
    ("Dusty", ("table_1",)): False,
    ("Soaked", ("cloth_1",)): True,   # doesn't matter -- goal marks this "unknown"
}
success, unmet = clean_table.check_goal(world_after_success)
print(f"rollout 1 -- success: {success}, unmet: {unmet}")

# world state after a failed rollout (table still dusty)
world_after_failure = {
    ("Dusty", ("table_1",)): True,
    ("Soaked", ("cloth_1",)): True,
}
success, unmet = clean_table.check_goal(world_after_failure)
print(f"rollout 2 -- success: {success}, unmet: {unmet}")

## 13. Code: a minimal initial-condition sampler

Section 5 mentioned that OmniGibson can generate "infinite valid physical configurations that satisfy an activity's initial conditions." Here's the core idea in miniature: given an activity's `init_predicates` (some of which involve *relational* placement, like "cloth on table"), repeatedly sample candidate object poses/states until every predicate is satisfied, or give up after a budget — the same rejection-sampling pattern the real system uses at much larger scale, with a real physics/collision check in place of our toy `region_contains` stand-in.

In [ ]:
def sample_valid_initial_state(activity: "ActivityDefinition",
                                 candidate_regions: dict,
                                 rng: np.random.Generator,
                                 max_attempts: int = 200) -> dict | None:
    """Rejection-sample object placements until every relational init predicate
    of type On(obj, region) is satisfied, or give up after `max_attempts`.

    Parameters
    ----------
    activity : ActivityDefinition
        Must have init_predicates of the form Predicate("On", (obj, region)).
    candidate_regions : dict[str, list[str]]
        For each object instance, the list of region names it's physically
        allowed to be sampled into (a stand-in for OmniGibson's real
        collision-aware region sampling).
    """
    on_predicates = [p for p in activity.init_predicates if p.name == "On"]

    for attempt in range(max_attempts):
        placement = {}
        for pred in on_predicates:
            obj, target_region = pred.args
            allowed = candidate_regions.get(obj, [])
            if target_region not in allowed:
                continue  # this predicate can never be satisfied with this object -- skip
            placement[obj] = target_region

        # accept only if every init predicate actually got placed as required
        satisfied = all(placement.get(p.args[0]) == p.args[1] for p in on_predicates)
        if satisfied:
            return {"attempt": attempt, "placement": placement}

    return None  # exhausted budget -- this is what a real sampler would report as unsolvable


# --- toy activity: a cloth must start "On" the table ---
init_activity = ActivityDefinition(
    name="CleanTableInit",
    objects={"cloth_1": "rag.n.01", "table_1": "table.n.02"},
    init_predicates=[Predicate("On", ("cloth_1", "table_1"))],
)
regions = {"cloth_1": ["table_1", "floor_1", "counter_1"]}

rng = np.random.default_rng(SEED)
result = sample_valid_initial_state(init_activity, regions, rng)
print("Sampled a valid initial state:", result)

# an unsolvable case: cloth is only ever allowed on the floor, never the table
unsolvable_regions = {"cloth_1": ["floor_1"]}
result_fail = sample_valid_initial_state(init_activity, unsolvable_regions, rng, max_attempts=50)
print("Unsolvable case result:", result_fail)

**Extending the toy sampler to a real sampling function** — Section 5.1 gave the exact `Open(o)` and `Cooked(o)` sampling rules. Here they are implemented directly against the `ObjectPhysicalState`/joint-state model, rather than the region-only placement toy above:

In [ ]:
def sample_open(joint_position_lower: float, joint_position_upper: float,
                 value: bool, rng: np.random.Generator, openness_threshold_frac: float = 0.05) -> float:
    """Real Open(o) sampling rule from Table A.9: for a single relevant joint,
    sample a position on the correct side of the 5%-of-range openness threshold.
    """
    threshold = joint_position_lower + openness_threshold_frac * (joint_position_upper - joint_position_lower)
    if value:
        return rng.uniform(threshold, joint_position_upper)
    return rng.uniform(joint_position_lower, threshold)


def check_open(joint_position: float, joint_position_lower: float, joint_position_upper: float,
               openness_threshold_frac: float = 0.05) -> bool:
    """The matching checking function: is the sampled/simulated joint position open?"""
    threshold = joint_position_lower + openness_threshold_frac * (joint_position_upper - joint_position_lower)
    return joint_position > threshold


rng = np.random.default_rng(SEED)
lower, upper = 0.0, 1.2  # a drawer's joint range, in meters of extension

for target in [True, False]:
    sampled_q = sample_open(lower, upper, target, rng)
    is_open = check_open(sampled_q, lower, upper)
    print(f"sampled Open(drawer) = {target}  ->  joint position = {sampled_q:.3f} m  ->  check_open() = {is_open}")

# reproduce Section 5.1's Cooked(o) sampling rule numerically as well
crab = ObjectPhysicalState("crab_1")
sample_cooked(crab, True, t_cooked=63.0)   # the paper's real annotated cook temperature for crab
print(f"\nsampled Cooked(crab)=True at t_cooked=63C -> crab.max_temperature = {crab.max_temperature:.1f}C"
      f" -> check_cooked = {check_cooked(crab, t_cooked=63.0)}")

**Visualizing the sampler** — how the rejection-sampling acceptance rate changes as an object's allowed-region list shrinks (i.e., as the scene gets more constrained):

In [ ]:
def acceptance_rate_vs_constraint(n_allowed_regions_options, total_regions=10, n_trials=300, rng=None):
    """For a single-object On(obj, target_region) predicate, estimate how often
    rejection sampling succeeds within a fixed attempt budget as a function of
    how many of `total_regions` candidate regions the object is actually allowed
    to occupy -- a minimal, quantitative version of what 'a more constrained
    scene is harder to sample a valid initial state for' means in practice.
    """
    rng = rng or np.random.default_rng(SEED)
    rates = []
    for n_allowed in n_allowed_regions_options:
        successes = 0
        for _ in range(n_trials):
            allowed = rng.choice(total_regions, size=n_allowed, replace=False)
            target = 0  # the target region index the goal actually requires
            # a single random draw represents one sampling attempt's candidate placement
            candidate = rng.choice(total_regions)
            successes += int((target in allowed) and (candidate == target))
        rates.append(successes / n_trials)
    return rates


allowed_options = [1, 2, 3, 5, 7, 10]
rates = acceptance_rate_vs_constraint(allowed_options)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(allowed_options, rates, "o-", color="#4fc3f7")
ax.set_xlabel("# regions the object is allowed to be sampled into")
ax.set_ylabel("single-attempt acceptance rate")
ax.set_title("Why 'more object placement freedom' speeds up initial-state sampling")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Code: an action-primitive interface sketch

Section 7 described RL-Prim.'s action space: a **discrete choice of primitive** (pick, place, push, navigate, dip, wipe) applied to a target object, resolved by sampling-based motion planning under the hood. Below is a minimal, runnable sketch of that interface — a `PrimitiveController` that receives `(primitive_name, target_object)` and dispatches to a per-primitive feasibility check, mirroring the paper's own simplification (Section 7): check only whether the **final configuration** is feasible, and if so, jump straight there rather than simulating the full trajectory. This is exactly the assumption that Section 8's ablation showed matters enormously for grasping and comparatively little for motion execution — worth having the actual mechanism in front of you when that ablation result comes up in discussion.

In [ ]:
from enum import Enum


class Primitive(Enum):
    PICK = "pick"
    PLACE = "place"
    PUSH = "push"
    NAVIGATE = "navigate"
    DIP = "dip"     # dip a held object (e.g. a cloth) into a fluid source
    WIPE = "wipe"    # wipe a held object across a target surface


class PrimitiveController:
    """Toy stand-in for OmniGibson's action-primitive execution layer. Each
    primitive's feasibility check is a simplification of what a real
    sampling-based motion planner would verify (reachability, collision-freeness
    of the *final* configuration only -- see Section 7/8's discussion of why
    this specific simplification matters so much for grasping realism).
    """

    def __init__(self, rng: np.random.Generator, grasp_success_prob: float = 0.9):
        self.rng = rng
        self.grasp_success_prob = grasp_success_prob  # tune down to see Sec. 8's finding

    def execute(self, primitive: Primitive, target: str, held_object: str | None = None) -> dict:
        if primitive == Primitive.PICK:
            # this is exactly the "assistive grasp" simplification: succeeds with
            # fixed probability rather than running full physics-based grasping
            success = self.rng.random() < self.grasp_success_prob
            return {"primitive": primitive.value, "target": target, "success": success,
                    "note": "assistive grasp (simplified)" if success else "grasp failed"}

        if primitive == Primitive.NAVIGATE:
            # motion execution simplification: teleport-to-endpoint if reachable
            reachable = self.rng.random() < 0.97  # motion planning rarely fails outright
            return {"primitive": primitive.value, "target": target, "success": reachable}

        # place / push / dip / wipe: assume feasible if something is currently held
        success = held_object is not None
        return {"primitive": primitive.value, "target": target, "success": success,
                "held_object": held_object}


controller = PrimitiveController(rng=np.random.default_rng(SEED), grasp_success_prob=0.9)

# a minimal CleanTable-style primitive sequence
log = []
log.append(controller.execute(Primitive.NAVIGATE, target="table_1"))
log.append(controller.execute(Primitive.PICK, target="cloth_1"))
log.append(controller.execute(Primitive.DIP, target="sink_1", held_object="cloth_1"))
log.append(controller.execute(Primitive.WIPE, target="table_1", held_object="cloth_1"))

for step in log:
    print(step)

overall_success = all(s["success"] for s in log)
print(f"\nfull sequence success: {overall_success}")
print("\nTry lowering grasp_success_prob to ~0.4 and re-running -- notice how quickly")
print("overall sequence success collapses, since every downstream primitive after a")
print("failed PICK is meaningless. This is a miniature, quantitative echo of Section 8's")
print("finding that fully physics-based grasping (a much lower effective success rate")
print("than this toy's fixed probability) is where most of the real difficulty hides.")

**Making the grasping finding quantitative, not just anecdotal** — sweep `grasp_success_prob` across a full range and Monte Carlo the whole primitive sequence at each value, so the collapse from Section 8's ablation shows up as a curve you derived yourself rather than a number you read off a table:

In [ ]:
def sequence_success_rate(grasp_prob: float, n_trials: int = 400, rng=None) -> float:
    """Monte Carlo the CleanTable-style primitive sequence (navigate -> pick ->
    dip -> wipe) `n_trials` times at a given grasp success probability, and
    return the fraction of trials where every primitive in the sequence succeeded.
    """
    rng = rng or np.random.default_rng(SEED)
    successes = 0
    for _ in range(n_trials):
        ctrl = PrimitiveController(rng=rng, grasp_success_prob=grasp_prob)
        steps = [
            ctrl.execute(Primitive.NAVIGATE, target="table_1"),
            ctrl.execute(Primitive.PICK, target="cloth_1"),
        ]
        held = "cloth_1" if steps[-1]["success"] else None
        steps.append(ctrl.execute(Primitive.DIP, target="sink_1", held_object=held))
        steps.append(ctrl.execute(Primitive.WIPE, target="table_1", held_object=held))
        successes += int(all(s["success"] for s in steps))
    return successes / n_trials


grasp_probs = np.linspace(0.05, 1.0, 20)
rng = np.random.default_rng(SEED)
seq_rates = [sequence_success_rate(p, rng=rng) for p in grasp_probs]

fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(grasp_probs, seq_rates, "o-", color="#ef5350")
ax.axvline(0.9, color="#4fc3f7", linestyle="--", alpha=0.7, label="this notebook's default (0.9)")
ax.set_xlabel("single-step grasp success probability")
ax.set_ylabel("full 4-step sequence success rate")
ax.set_title("Why grasping reliability dominates long-horizon task success")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Even a fairly reliable grasp (e.g. 70% per attempt) only yields",
      f"{sequence_success_rate(0.7, rng=rng):.0%} success across this 4-step",
      "sequence -- and BEHAVIOR-1K's real activities need far more than 4 steps",
      "(CollectTrash needs 16+). This compounding is exactly why Section 8's",
      "ablation shows such a dramatic collapse once grasping is no longer",
      "artificially reliable, and why longer-horizon activities are hit hardest.")

**Interactive version** — if your notebook environment supports `ipywidgets` (Colab and standard Jupyter both do), drag the slider below to explore the same relationship live rather than reading it off the static curve above:

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    def _interactive_sweep(grasp_success_prob=0.9, n_steps=4):
        rng = np.random.default_rng(SEED)
        seq_rate = sequence_success_rate(grasp_success_prob, n_trials=500, rng=rng)
        compounded = grasp_success_prob ** n_steps  # naive independent-steps comparison

        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.bar(["Monte Carlo\n(this sequence)", f"Naive p^{n_steps}\n(independent steps)"],
               [seq_rate, compounded], color=["#ef5350", "#4fc3f7"])
        ax.set_ylim(0, 1)
        ax.set_ylabel("sequence success rate")
        ax.set_title(f"grasp_success_prob = {grasp_success_prob:.2f}")
        plt.tight_layout()
        plt.show()

    widgets.interact(
        _interactive_sweep,
        grasp_success_prob=widgets.FloatSlider(value=0.9, min=0.05, max=1.0, step=0.05,
                                                description="grasp p:"),
        n_steps=widgets.IntSlider(value=4, min=1, max=20, step=1, description="# steps:"),
    )
except ImportError:
    print("ipywidgets isn't available in this environment -- install with")
    print("`pip install ipywidgets` to get the live slider. The static sweep")
    print("above already shows the same relationship without it.")

## 15. Code: reproducing the survey's diversity statistic (Gini index)

Section 2 reported a Gini index of 0.158 over the survey's 2,090 activity preference scores, as a single-number summary of how unevenly people's automation-desire is spread across activities. The Gini index is worth having in your general toolkit — it's the standard inequality/dispersion measure from economics (originally for income distributions), and it generalizes to "how concentrated is any ranked, non-negative distribution" — group meeting presentations on any survey-driven benchmark will likely need you to explain it on the spot.

In [ ]:
def gini_index(values: np.ndarray) -> float:
    """Gini index of a non-negative array: 0 = perfectly even distribution,
    1 = maximally concentrated (all mass on one item). Computed via the standard
    mean-absolute-difference formula, normalized by twice the mean.
    """
    values = np.sort(np.asarray(values, dtype=float))
    n = len(values)
    if n == 0 or values.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * values) - (n + 1) * np.sum(values)) / (n * np.sum(values)))


rng = np.random.default_rng(SEED)

# toy scenario A: scores roughly uniform across activities (low dispersion)
uniform_scores = rng.uniform(4.5, 5.5, size=2090)

# toy scenario B: scores with a long tail of "very wanted" chores (like the real
# survey's tedious-chores-score-highest pattern), calibrated to land near the
# paper's reported range (1.9-9.3, mean 5.16, Gini 0.158)
skewed_scores = np.clip(rng.gamma(shape=6.0, scale=0.86, size=2090) + 1.0, 1.0, 9.5)

print(f"Uniform toy scenario  -- mean: {uniform_scores.mean():.2f}, Gini: {gini_index(uniform_scores):.3f}")
print(f"Skewed toy scenario   -- mean: {skewed_scores.mean():.2f}, Gini: {gini_index(skewed_scores):.3f}")
print(f"(Paper's actual survey -- mean: 5.16, Gini: 0.158, over 2,090 activities)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(uniform_scores, bins=30, color="#4fc3f7", alpha=0.85)
axes[0].set_title(f"Low-dispersion toy scenario (Gini={gini_index(uniform_scores):.3f})")
axes[0].set_xlabel("preference score")

axes[1].hist(skewed_scores, bins=30, color="#ffca28", alpha=0.85)
axes[1].set_title(f"Skewed toy scenario (Gini={gini_index(skewed_scores):.3f})")
axes[1].set_xlabel("preference score")

plt.tight_layout()
plt.show()

In [ ]:
def lorenz_curve(values: np.ndarray):
    """Return (cumulative population share, cumulative value share) for the
    classic Lorenz-curve visualization of inequality -- the geometric object
    the Gini index is literally the area between and the diagonal, times two.
    """
    sorted_vals = np.sort(np.asarray(values, dtype=float))
    cum_vals = np.cumsum(sorted_vals)
    cum_share = np.insert(cum_vals / cum_vals[-1], 0, 0)
    pop_share = np.linspace(0, 1, len(cum_share))
    return pop_share, cum_share


fig, ax = plt.subplots(figsize=(6.5, 6))
pop_share, cum_share = lorenz_curve(skewed_scores)
ax.plot(pop_share, cum_share, color="#ffca28", linewidth=2, label=f"Lorenz curve (Gini={gini_index(skewed_scores):.3f})")
ax.plot([0, 1], [0, 1], color="#888888", linestyle="--", label="perfect equality")
ax.fill_between(pop_share, pop_share, cum_share, color="#ffca28", alpha=0.25)
ax.set_xlabel("cumulative share of activities (sorted lowest -> highest score)")
ax.set_ylabel("cumulative share of total preference score")
ax.set_title("Lorenz curve: the Gini index IS (2 x the shaded area)")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print("The shaded gap between the diagonal (perfect equality -- every activity",
      "equally wanted) and the yellow curve (actual distribution) is exactly what",
      "the Gini index quantifies. A flatter curve, closer to the diagonal, means a",
      "lower Gini and a more even spread of 'how much people want this automated.'")

## 16. BEHAVIOR-1K vs. LIBERO vs. CALVIN vs. RLBench

Since you've already built tutorials on the other three, here's where BEHAVIOR-1K sits relative to them — worth having this table in mind any time a comparison across benchmarks comes up in a meeting:

| | **BEHAVIOR-1K** | **LIBERO** | **CALVIN** | **RLBench** |
|---|---|---|---|---|
| Primary axis studied | Breadth + realism of long-horizon household activities, grounded in human-preference survey | Lifelong learning: declarative vs. procedural knowledge transfer | Long-horizon, language-conditioned skill chaining in one continuous scene | Single/multi-task manipulation skill breadth |
| # tasks/activities | 1,000 | 130 | small atomic-skill vocabulary, chained | 100+ |
| Scene diversity | 50 scenes across 8 scene types (houses, offices, restaurants, stores...) | tabletop only | one continuous simulated apartment | tabletop only |
| Simulator | OmniGibson (NVIDIA Omniverse/PhysX 5) | MuJoCo via `robosuite` | PyBullet | CoppeliaSim |
| Physical realism | rigid + deformable + fluid + thermal | rigid body only | rigid body only | rigid body only |
| Hardware requirement | **NVIDIA RTX GPU required**, no CPU fallback | CPU-only (MuJoCo) | CPU-friendly | CPU-friendly (typically) |
| Task specification | BDDL (extended: substances, 3-valued predicates, composition) | BDDL (LIBERO subset) | scripted language-annotated skill chains | hand-coded per-task Python |
| Task-list provenance | 1,461-person human-preference survey | researcher-designed, structured for controlled knowledge-transfer shift | researcher-designed | researcher-designed |

**The one-sentence framing worth internalizing:** LIBERO asks "can a policy learn a *sequence* of tasks without forgetting," CALVIN and RLBench ask "can a policy do *many* things," and BEHAVIOR-1K asks a different question from all three — "can a policy do the things *people actually want*, in scenes and with physics realistic enough that success in simulation says something about success in a real home." That's a benchmark-design axis (task provenance + simulation realism) rather than a learning-algorithm axis, and it's the reason BEHAVIOR-1K's baselines (Section 7) look so much weaker than what you'd see reported on LIBERO or RLBench — the activities are harder *by design*, not because the field's methods regressed.

## 17. Discussion questions

**Survey methodology & benchmark design**

1. The activity pool was filtered for simulation feasibility *before* the survey ran (Section 2.1). What does this do to the interpretation of "BEHAVIOR-1K reflects what humans want robots to do"? Is there a version of this claim that's still true, and a version that overclaims?
2. The survey found a Gini index of 0.158 — described as "large statistical dispersion." Given the Gini index runs from 0 (perfectly even) to 1 (maximally concentrated), do you agree 0.158 supports "large" dispersion? What would you want to know (e.g., a reference Gini for a familiar distribution) to judge that claim rather than take it on faith?
3. Properties are annotated at leaf synsets only, then propagated upward as an **intersection** (Section 11's code). Construct a scenario where this design choice causes an activity definition to be *rejected as impossible* even though a human would consider it obviously achievable. What's the fix — annotate more leaf synsets, or restructure the hierarchy?

**Simulation & realism**

4. OmniGibson trades rendering speed (60fps vs. iGibson 2.0's ~100fps) for photorealism. For which of the three evaluated activities (CollectTrash, StoreDecoration, CleanTable) do you think that tradeoff matters most for a learning algorithm, and why?
5. The Transition Machine (Section 3/5) symbolically swaps object states for processes that are too complex to fully physically simulate (dough → pie). What's a concrete case where this symbolic shortcut could teach a policy something *wrong* about the physical world — i.e., where the shortcut's abstraction leaks into what the agent learns to expect?

**Results & sim-to-real**

6. RL-VMC (raw visuomotor control) gets 0% success on all three activities, while primitive-based RL reaches 42–88%. If a future paper reports a big RL-VMC success-rate improvement on BEHAVIOR-1K, what would you want to check before believing it represents genuine progress rather than, say, task selection or reward shaping?
7. The grasping-simplification ablation (Section 8) shows fully physics-based grasping collapses performance dramatically, while removing the motion-execution simplification barely changes it. Given that finding, if you were choosing where to spend a limited research budget to improve BEHAVIOR-1K baseline performance, would you prioritize better motion planning or better grasping? Defend your answer using the ablation, not intuition alone.
8. The real-robot study attributes some failures to "navigation inaccuracy in a previous timestep" leaving the base badly positioned for manipulation — an error class absent from simulation entirely, because sim assumes perfect localization. Propose one modification to the simulated evaluation protocol that would surface this failure mode *before* going to real hardware.

**Comparative**

9. If you were designing a follow-up to BEHAVIOR-1K that reused its survey methodology but targeted a different setting (e.g. warehouse work, elder care, restaurant kitchens), what would you need to change about the survey population, question wording, or filtering criteria — and what would you keep identical to preserve comparability?

**Technical / quantitative (new)**

10. PPO's entropy coefficient is 0.0 in these experiments, while SAC's entire objective (Eq. 2) is built around maximizing entropy. Given RL-VMC (SAC) fails completely and RL-Prim. (PPO) doesn't, how would you design a controlled experiment to separate "the action space made the problem easy" from "the exploration mechanism made the problem easy"?
11. Table A.10's performance ablation shows fluid/cloth simulation cost scales with scene object count, while removing the robot buys comparatively little. If you were optimizing OmniGibson for a specific research use case that never involves fluids or cloth, which feature would you disable first, and does the answer change between a small scene (`Rs_int`) and a large one (`house_single_floor`)?
12. Every logical-predicate threshold in Section 5.1 ($w_{soaked}=50$, $w_{filled}=0.5$, $T_{onfire}=300°C$) is a hand-set default, configurable per object category. Pick one threshold and describe a concrete object for which the default would give a physically wrong answer, and what re-annotation would fix it.
13. The three-valued-predicate mechanism (Section 4.1) compiles `not(Filled(x))` to `Empty(x)` — a separately-defined predicate, not literally "not filled." Construct a world state where this distinction changes whether an activity's goal is judged satisfied, compared to a naive Boolean-negation implementation.

## References

- Li, C., Zhang, R., Wong, J., Gokmen, C., Srivastava, S., Martín-Martín, R., Wang, C., Levine, G., Ai, W., et al. (2024). *BEHAVIOR-1K: A Human-Centered, Embodied AI Benchmark with 1,000 Everyday Activities and Realistic Simulation.* arXiv:2403.09227.
- Srivastava, S., Li, C., Lingelbach, M., Martín-Martín, R., Xia, F., Vainio, K.E., Lian, Z., Gokmen, C., Buch, S., Liu, K., et al. (2022). *BEHAVIOR: Benchmark for Everyday Household Activities in Virtual, Interactive, and Ecological Environments.* CoRL — the predecessor, BEHAVIOR-100.
- Official repository (BEHAVIOR-1K / OmniGibson): [github.com/StanfordVL/BEHAVIOR-1K](https://github.com/StanfordVL/BEHAVIOR-1K)
- Documentation: [behavior.stanford.edu](https://behavior.stanford.edu)
- Fox, M. & Long, D. (2003). *PDDL2.1: An Extension to PDDL for Expressing Temporal Planning Domains* — the predicate-logic planning-language lineage BDDL descends from.
- Miller, G. A. (1995). *WordNet: A Lexical Database for English.* Communications of the ACM — the synset hierarchy underlying the object taxonomy in Section 3/11.
- Li, C., Xia, F., Martín-Martín, R., Lingelbach, M., Srivastava, S., Shen, B., Vainio, K.E., Gokmen, C., Dharan, G., Jain, T., et al. (2021). *iGibson 2.0: Object-Centric Simulation for Robot Learning of Everyday Household Tasks.* CoRL — the predecessor simulator OmniGibson replaces.
- Related benchmarks discussed in Section 16: LIBERO (Liu et al., 2023), CALVIN (Mees et al., 2022), RLBench (James et al., 2020) — see your earlier `xulabs/edu` tutorials on these.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). *Proximal Policy Optimization Algorithms.* arXiv:1707.06347 -- the exact PPO objective used by RL-Prim./RL-Prim.Hist. (Sec. 7.1).
- Haarnoja, T., Zhou, A., Abbeel, P., & Levine, S. (2018). *Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor.* ICML -- the exact SAC objective used by RL-VMC (Sec. 7.1).